# 07 · Policy gradients: REINFORCE with a baseline

**Goal:** optimize a policy $\pi_\theta$ *directly*, by gradient ascent on expected return, instead of learning values first.

**Policy-gradient theorem (episodic form):**

$$\nabla_\theta J(\theta)=\mathbb E_{\pi_\theta}\Big[\sum_t \nabla_\theta\log\pi_\theta(a_t|s_t)\,G_t\Big],\qquad G_t=\sum_{k\ge t}\gamma^{k-t}r_k .$$

Subtracting a state-dependent **baseline** $b(s_t)$ from $G_t$ leaves the expectation unchanged (because $\mathbb E_a[\nabla\log\pi(a|s)]=0$) but can greatly reduce variance.

For a tabular softmax policy $\pi_\theta(a|s)=\mathrm{softmax}(\theta_{s,:})_a$:

$$\nabla_{\theta_{s,:}}\log\pi(a|s)=\mathbf e_a-\pi(\cdot|s).$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 4x4 gridworld, actions: 0=up 1=right 2=down 3=left, terminals in two opposite corners,
# reward -1 per step (so the optimal policy takes the shortest path to a corner).
N = 4
S = N * N
A = 4
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]
TERMINALS = {0, S - 1}

P = np.zeros((S, A, S))        # P[s, a, s'] transition probabilities
R = np.full((S, A), -1.0)      # R[s, a] expected reward
for s in range(S):
    if s in TERMINALS:
        P[s, :, s] = 1.0
        R[s, :] = 0.0
        continue
    r, c = divmod(s, N)
    for a, (dr, dc) in enumerate(MOVES):
        r2, c2 = r + dr, c + dc
        if not (0 <= r2 < N and 0 <= c2 < N):
            r2, c2 = r, c                      # bumping into a wall = stay put
        P[s, a, r2 * N + c2] = 1.0
assert np.allclose(P.sum(axis=2), 1.0)         # every row is a probability distribution

In [ ]:
gamma = 0.99
rng = np.random.default_rng(0)
nonterminal = np.array([s for s in range(S) if s not in TERMINALS])

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def rollout(theta, s0, rng, max_steps=60):
    states, actions, rewards = [], [], []
    s = s0
    for _ in range(max_steps):
        if s in TERMINALS:
            break
        a = rng.choice(A, p=softmax(theta[s]))
        s2 = int(np.argmax(P[s, a]))                # deterministic grid
        states.append(s); actions.append(a); rewards.append(R[s, a])
        s = s2
    return states, actions, rewards

def returns(rewards):
    G = np.zeros(len(rewards)); run = 0.0
    for t in reversed(range(len(rewards))):
        run = rewards[t] + gamma * run
        G[t] = run
    return G

## Train

In [ ]:
def reinforce(episodes=6000, lr=0.1, seed=0, use_baseline=True):
    rng = np.random.default_rng(seed)
    theta = np.zeros((S, A))
    b = np.zeros(S)                  # running-mean baseline per state
    lengths = []
    for ep in range(episodes):
        s0 = rng.choice(nonterminal)
        st, ac, rw = rollout(theta, s0, rng)
        G = returns(rw)
        for t, (s, a) in enumerate(zip(st, ac)):
            adv = G[t] - (b[s] if use_baseline else 0.0)
            g = -softmax(theta[s]); g[a] += 1.0              # grad log pi(a|s)
            theta[s] += lr * (gamma ** t) * adv * g
            b[s] += 0.05 * (G[t] - b[s])
        lengths.append(len(rw))
    return theta, np.array(lengths)

theta, lengths = reinforce()
print("mean episode length, first 200:", lengths[:200].mean().round(2),
      "  last 200:", lengths[-200:].mean().round(2))

## Evaluate the learned policy against the optimal path length

In [ ]:
# optimal number of steps from each state to the nearest terminal (BFS = Manhattan distance here)
def dist_to_terminal(s):
    r, c = divmod(s, N)
    return min(abs(r - tr) + abs(c - tc) for tr, tc in (divmod(t, N) for t in TERMINALS))

steps_taken, steps_opt = [], []
for s0 in nonterminal:
    s, k = s0, 0
    while s not in TERMINALS and k < 60:
        a = int(np.argmax(theta[s]))                 # greedy w.r.t. learned preferences
        s = int(np.argmax(P[s, a])); k += 1
    steps_taken.append(k); steps_opt.append(dist_to_terminal(s0))
steps_taken, steps_opt = np.array(steps_taken), np.array(steps_opt)
print("steps taken  :", steps_taken)
print("steps optimal:", steps_opt)
assert np.all(steps_taken < 60)                      # always reaches a terminal
assert steps_taken.mean() <= 1.25 * steps_opt.mean()

plt.plot(np.convolve(lengths, np.ones(100) / 100, mode="valid"))
plt.xlabel("episode"); plt.ylabel("episode length (100-ep average)"); plt.show()

## Compare with and without the baseline

Run the same experiment over several seeds and compare how quickly episode length falls. The gradient estimates have the same mean, but the baseline version should have visibly lower variance.

In [ ]:
def early_length(use_baseline, seeds=range(5), episodes=1500):
    vals = []
    for sd in seeds:
        _, L = reinforce(episodes=episodes, seed=sd, use_baseline=use_baseline)
        vals.append(L[-300:].mean())
    return np.array(vals)

with_b = early_length(True)
without_b = early_length(False)
print("last-300-episode mean length, with baseline   :", with_b.round(2), " avg", with_b.mean().round(2))
print("last-300-episode mean length, without baseline:", without_b.round(2), " avg", without_b.mean().round(2))

## Exercises

1. Prove $\mathbb E_{a\sim\pi}[\nabla_\theta\log\pi(a|s)\,b(s)]=0$ using $\sum_a\pi(a|s)=1$.
2. Replace the running-mean baseline by a learned $V_\phi(s)$ updated by TD(0). This is **actor-critic**.
3. Swap tabular softmax for a small NumPy network (reuse notebook 04's backprop) on a continuous-state task of your choice.